In [ ]:
# Cell 1: Load and explore label data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the concepts CSV
path = "/kaggle/input/datasets/quocthai912/imageclefdatasetstrain/dev_concept/concepts.csv" 
df = pd.read_csv(path)

# Ensure CUIs column is string; replace NaN with empty string
df["CUIs"] = df["CUIs"].fillna("").astype(str)

# Parse label list: split by ";" and strip whitespace
df["label_list"] = df["CUIs"].apply(
    lambda x: [c.strip() for c in x.split(";") if c.strip() != ""]
)

# Count unique labels
all_labels = [label for labels in df["label_list"] for label in labels]
unique_labels = set(all_labels)
num_unique_labels = len(unique_labels)

print(f"Total unique concepts: {num_unique_labels}")

# Average number of labels per image
avg_labels_per_image = df["label_list"].apply(len).mean()
print(f"Average labels per image: {avg_labels_per_image:.4f}")

# Label frequency
label_counts = pd.Series(all_labels).value_counts()

# Plot top 20 most frequent labels
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

top_20_most = label_counts.head(20).sort_values(ascending=True)  
plt.figure(figsize=(14, 10))
sns.barplot(x=top_20_most.values, y=top_20_most.index, palette="viridis")
plt.title("Top 20 Most Frequent Labels", fontsize=16, weight="bold")
plt.xlabel("Frequency", fontsize=12)
plt.ylabel("Labels (CUI)", fontsize=12)
plt.tight_layout()
plt.show()

# Plot top 20 least frequent labels
top_20_least = label_counts.tail(20).sort_values(ascending=True)
plt.figure(figsize=(14, 10))
sns.barplot(x=top_20_least.values, y=top_20_least.index, palette="magma")
plt.title("Top 20 Least Frequent Labels", fontsize=16, weight="bold")
plt.xlabel("Frequency", fontsize=12)
plt.ylabel("Labels (CUI)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 2: Dataset, transforms, and DataLoader (full development set)
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Config
CSV_PATH = "/kaggle/input/datasets/quocthai912/imageclefdatasetstrain/dev_concept/concepts.csv"
IMAGE_DIR = "/kaggle/input/datasets/quocthai912/imageclefdatasetstrain/dev_concept/images"

BATCH_SIZE = 32
NUM_WORKERS = 2
PIN_MEMORY = True

# Training transforms with light augmentation
train_transforms = A.Compose([
    A.Resize(224, 224),
    A.ShiftScaleRotate(
        shift_limit=0.05,   # max 5% translation
        scale_limit=0.05,   # max 5% zoom
        rotate_limit=10,    # max 10 degrees rotation
        p=0.5
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0.2,  # 20% brightness jitter
        contrast_limit=0.2,    # 20% contrast jitter
        p=0.5
    ),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),   # ImageNet mean
        std=(0.229, 0.224, 0.225),    # ImageNet std
        max_pixel_value=255.0
    ),
    ToTensorV2()
])

# Validation/test transforms (no augmentation)
valid_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),   
        std=(0.229, 0.224, 0.225),    
        max_pixel_value=255.0
    ),
    ToTensorV2()
])

class MedicalDataset(Dataset):
    def __init__(self, image_paths, labels, transforms=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transforms = transforms

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Cannot read image at path: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transforms:
            image = self.transforms(image=image)["image"]
            
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, label

# Load CSV and parse labels
df = pd.read_csv(CSV_PATH)
df["ID"] = df["ID"].astype(str).str.strip()
df["CUIs"] = df["CUIs"].fillna("").astype(str)
df["label_list"] = df["CUIs"].apply(lambda s: [x.strip() for x in s.split(";") if x.strip()])

mlb = MultiLabelBinarizer()
labels_multi_hot = mlb.fit_transform(df["label_list"])

# Scan image directory and match with CSV IDs
all_files = os.listdir(IMAGE_DIR)
id2path = {}
for fname in all_files:
    if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
        image_id = os.path.splitext(fname)[0]
        id2path[image_id] = os.path.join(IMAGE_DIR, fname)

df["image_path"] = df["ID"].map(id2path)

found_mask = df["image_path"].notna().values
df = df.loc[found_mask].reset_index(drop=True)
labels_multi_hot = labels_multi_hot[found_mask]

# Use the full development set for training (full-fit strategy)
train_image_paths = df["image_path"].tolist()
train_labels = labels_multi_hot

train_dataset = MedicalDataset(train_image_paths, train_labels, transforms=train_transforms)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=NUM_WORKERS, 
    pin_memory=PIN_MEMORY
)

print("-" * 40)
print("TRAINING SET:")
print(f"Total images loaded: {len(train_dataset)}")
print(f"Number of classes (MLB): {len(mlb.classes_)}")
print("-" * 40)


In [ ]:
# Cell 3: Sanity check — sample one batch from the training loader
sample_images, sample_labels = next(iter(train_loader))

print("Image batch shape:", sample_images.shape)
print("Label batch shape:", sample_labels.shape)


In [ ]:
# Cell 4: Build model, loss function, optimizer, and scheduler
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# Config
NUM_CLASSES = 2646
MODEL_NAME = "densenet121"
PRETRAINED = True
LR = 2e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 6

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build DenseNet121 with timm
model = timm.create_model(
    MODEL_NAME,
    pretrained=PRETRAINED,
    num_classes=NUM_CLASSES,
    drop_rate=0.2
).to(device)

# Asymmetric Loss (ASL) — suited for multi-label long-tail classification
class AsymmetricLossOptimized(nn.Module):
    def __init__(
        self,
        gamma_neg: float = 4.0,
        gamma_pos: float = 1.0,
        clip: float = 0.05,
        eps: float = 1e-8,
        disable_torch_grad_focal_loss: bool = False
    ):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps
        self.disable_torch_grad_focal_loss = disable_torch_grad_focal_loss

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.float()

        # Probabilities
        x_sigmoid = torch.sigmoid(logits)
        xs_pos = x_sigmoid
        xs_neg = 1.0 - x_sigmoid

        # Asymmetric clipping
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)

        los_pos = targets * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps))
        loss = los_pos + los_neg

        # Asymmetric focusing
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(False)

            pt0 = xs_pos * targets
            pt1 = xs_neg * (1.0 - targets)
            pt = pt0 + pt1
            gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
            one_sided_w = torch.pow(1.0 - pt, gamma)

            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(True)

            loss *= one_sided_w

        return -loss.mean()

criterion = AsymmetricLossOptimized(
    gamma_neg=4.0,   
    gamma_pos=1.0,   
    clip=0.05
).to(device)

# AdamW optimizer and cosine annealing scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=EPOCHS, 
    eta_min=1e-6
)

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# Verify
final_layer = model.get_classifier()
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: {MODEL_NAME}")
print(f"Final classifier layer:\n{final_layer}")
print(f"Total parameters: {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")


In [ ]:
# Cell 5: Training loop (full-fit on entire development set)
import torch
from tqdm.auto import tqdm

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, desc="Training", leave=False, dynamic_ncols=True)

    for step, batch in enumerate(pbar):
        # Support both tuple/list and dict batch formats
        if isinstance(batch, (list, tuple)):
            images, targets = batch
        elif isinstance(batch, dict):
            images = batch["images"]
            targets = batch["targets"]
        else:
            raise ValueError("Unsupported batch format. Please use tuple/list or dict.")

        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)

        # Mixed precision forward pass
        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, targets)

        # Backward + optimizer step with GradScaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_item = loss.detach().item()
        running_loss += loss_item

        avg_so_far = running_loss / (step + 1)
        pbar.set_postfix(loss=f"{loss_item:.4f}", avg=f"{avg_so_far:.4f}")

    epoch_loss = running_loss / len(loader)
    return epoch_loss

# Train for NUM_EPOCHS; use a small number of epochs to reduce overfitting
NUM_EPOCHS = 6
best_train_loss = float("inf")

print("Starting full-fit training (100% of development data)...")

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=device, 
    )

    print(f"Epoch [{epoch}/{NUM_EPOCHS}] - Avg Train Loss: {train_loss:.6f}")
    
    scheduler.step()
    
    # Save checkpoint whenever training loss improves
    if train_loss < best_train_loss:
        best_train_loss = train_loss
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "best_train_loss": best_train_loss,
            },
            "best_model_densenet121.pth",
        )
        print(f"Saved best_model_densenet121.pth (best_train_loss={best_train_loss:.6f})")

print("Training complete.")


In [ ]:
# Cell 6: Inference on test set and generate submission file
import os
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("Running inference on test set...")

# Test set path
TEST_DIR = "/kaggle/input/datasets/quocthai912/imageclef-caption-2026-test-datasets/test/images"

class MedicalTestDataset(Dataset):
    def __init__(self, image_dir, transforms=None):
        self.image_dir = image_dir
        self.filenames = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        self.image_ids = [os.path.splitext(f)[0] for f in self.filenames]
        self.transforms = transforms

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.filenames[idx])
        image = cv2.imread(img_path)
        
        # Fall back to a black image if the file cannot be read
        if image is None:
            print(f"WARNING: Skipping unreadable image {self.filenames[idx]}")
            image = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
        if self.transforms:
            image = self.transforms(image=image)["image"]
        return image, self.image_ids[idx]

# Use valid_transforms defined in Cell 2 (no augmentation)
test_dataset = MedicalTestDataset(TEST_DIR, transforms=valid_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Found {len(test_dataset)} images in test set.")

# Load best DenseNet121 checkpoint
checkpoint = torch.load("/kaggle/input/datasets/quocthai912/model-densenet-121-final-version-1/best_model_densenet121.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded DenseNet121 weights from epoch {checkpoint['epoch']} (train loss: {checkpoint['best_train_loss']:.4f})")

# Run predictions
all_probs = []
all_test_ids = []

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Testing"):
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            logits = model(images)
            probs = torch.sigmoid(logits)
        
        all_probs.append(probs.cpu().numpy())
        all_test_ids.extend(ids)

all_probs = np.vstack(all_probs)

# Apply threshold found during development
THRESHOLD = 0.60
binary_preds = (all_probs >= THRESHOLD).astype(int)

# Decode predictions to CUI strings
predicted_cuis_tuple = mlb.inverse_transform(binary_preds)
predicted_cuis_str = [";".join(cuis) for cuis in predicted_cuis_tuple]

submission_df = pd.DataFrame({
    "ID": all_test_ids,
    "CUIs": predicted_cuis_str
})

# Sort by numeric suffix of the ID (e.g. ImageCLEFmedical_Caption_2026_test_12989 -> 12989)
submission_df['id_num'] = submission_df['ID'].str.extract(r'_(\d+)$').astype(int)
submission_df = submission_df.sort_values('id_num').drop(columns=['id_num']).reset_index(drop=True)

submission_df.to_csv("submission.csv", index=False)

print(f"submission.csv created — {len(submission_df)} predictions.")
print(submission_df.head())


In [ ]:
# Cell 7: Preview submission file
submission_df
